# 5장 4강 : 이터레이터와 제너레이터
## 학습 목표
- 이터러블과 이터레이터의 순회 메커니즘인 __iter__()와 __next__()를 설명할 수 있다.
- 제너레이터의 지연 평가 특성을 이해하고, 대용량 파일 가공 시의 메모리 점유율을 최적화할 수 있다.

### 1. 이터러블과 이터레이터의 동작 방식
---
#### 1.1 순회 가능한 객체(Iterable)와 이터레이터(Iterator)의 차이
- 이터러블(Iterable)의 정의
    - 리스트, 문자열, 딕셔너리처럼 내부 요소를 하나씩 꺼내어 순회할 수 있는 모든 객체를 의미
    -  이터러블 객체는 내부적으로 __iter__ 매직 메서드를 가지고 있어서, 이를 통해 이터레이터를 만들어낼 수 있습니다.

- 이터레이터(Iterator)의 포인터 역할
    - 이터러블 객체에서 요소를 순서대로 꺼내는 '비밀 화살표(포인터)' 역할을 하는 객체
    - 다음 요소를 가리키는 __next__ 매직 메서드를 구현하고 있어서, 한 번에 하나씩 데이터를 메모리에 불러와 처리하는 동작 방식을 가집니다.

- 수동 순회 제어
    - iter() 함수는 객체의 __iter__ 메서드를 실행하고, next() 함수는 이터레이터의 __next__ 메서드를 가동하여 원소를 수동 추출합니다. 꺼낼 원소가 더 이상 없으면 파이썬은 StopIteration 예외를 발생시켜 순회가 완료되었음을 식별합니다.


In [1]:
# 내장 리스트 객체(Iterable) 생성
fruits = ["사과", "바나나", "체리"]

# iter() 함수를 사용하여 이터레이터 객체 추출하기
fruits_iterator = iter(fruits)
print(f"이터레이터 객체 확인: {fruits_iterator}")

# next()를 호출하며 원소를 순차 수동 추출하기
print(next(fruits_iterator))  # 사과
print(next(fruits_iterator))  # 바나나
print(next(fruits_iterator))  # 체리

# 더 이상 꺼낼 원소가 없을 때 예외 발생 확인
# print(next(fruits_iterator))  # StopIteration 예외 발생

이터레이터 객체 확인: <list_iterator object at 0x000001E5C57992A0>
사과
바나나
체리


- fruits 리스트는 내부적으로 **__iter__** 메서드를 가진 이터러블 객체입니다.
- iter(fruits)가 실행되면, 현재 리스트의 첫 번째 원소 직전 위치를 기억하는 포인터를 품은 fruits_iterator 객체가 생성되어 메모리에 로드됩니다.
- next()를 호출할 때마다 포인터가 한 칸씩 이동하며 "사과", "바나나", "체리" 데이터를 차례대로 가져옵니다.
- 모든 원소를 전부 꺼낸 후 다시 next()를 호출하면, 내부 순서가 종료되었음을 알리는 **StopIteration** 예외를 도출하여 안전하게 반복을 마감합니다.

### 2. yield 키워드와 제너레이터의 지연 평가
---
- 일반적인 함수는 return을 만나면 연산된 최종 결과값을 반환하고 함수 공간 전체를 메모리에서 소멸시킵니다. 
    반면, yield 키워드를 사용하는 '제너레이터(Generator)' 함수는 데이터를 한 번에 모두 만들어서 메모리에 쌓아두지 않고, 호출될 때마다 값을 하나씩 생성하여 내보내는 '지연 평가(Lazy Evaluation)' 동작 방식을 취합니다. 제너레이터가 대용량 데이터를 처리할 때 리소스를 어떻게 아끼는지 알아보겠습니다.

#### 2.1 yield 키워드를 사용한 함수의 일시 정지 및 상태 보존
- 함수의 일시 정지
    - 제어 흐름 중에 yield 키워드를 만나면 함수는 그 시점의 연산 결과값을 함수 밖으로 반환함과 동시에, 자신의 실행 상태(지역 변수 값, 포인터 위치 등)를 메모리에 고스란히 얼려둔 채로 잠시 대기합니다.
- 상태 보존과 재개
    - 외부에서 다시 데이터를 요청(next())하면 함수는 처음부터 다시 실행되는 것이 아니라, 얼어있던 상태를 깨워 yield 바로 다음 줄부터 실행을 이어가는 독특한 동작 방식을 취합니다.

#### 2.2 지연 평가(Lazy Evaluation)와 일반 연산의 효율 비교
- 지연 평가의 개념
    - 데이터가 필요한 그 시점이 되어서야 비로소 계산을 수행하여 값을 생성하는 전략입니다.
- 메모리 점유율 최적화
    - 일반적인 리스트나 배열 연산은 100만 개의 원소가 있다면 100만 개를 모두 메모리에 한 번에 적재하므로 대용량 가공 시 컴퓨터 시스템에 무리를 줍니다. 반면 제너레이터는 yield를 통해 현재 처리할 단 하나의 데이터만 메모리에 올리므로, 기가바이트 단위의 가상 정수 데이터 구조나 끝이 없는 무한 수열 구조도 아주 적은 메모리만으로 안전하게 설계하고 구현할 수 있습니다.

In [ ]:
# 무한 수열 데이터 구조를 yield 기반 제너레이터 함수로 구현하기
def infinite_number_generator():
    number = 1
    while True:
        yield number  # 값을 반환하고 함수의 실행 상태를 메모리에 보관한 채 일시 정지
        number += 1   # 다음 호출 시 이 지점부터 실행 재개

# 제너레이터 객체 생성
num_gen = infinite_number_generator()
print(f"제너레이터 객체 타입: {num_gen}")

# 필요한 시점에만 next()로 하나씩 데이터를 추출하여 비교
print(next(num_gen))  # 1
print(next(num_gen))  # 2
print(next(num_gen))  # 3

- infinite_number_generator() 함수는 내부에 while True 반복 루프가 있지만, 호출되는 순간 코드를 전부 가동하지 않고 데이터를 뽑아낼 수 있는 제너레이터 객체만 반환하여 메모리를 보호합니다.
- next(num_gen)이 실행되면 함수 내부로 진입하여 number = 1을 세팅하고 yield 1을 만나 1을 반환하며 그 자리에 그대로 일시 정지합니다.
- 다음 next(num_gen)이 실행되면 정지했던 지점 바로 뒤인 number += 1이 연산되어 값이 2가 되고, 다시 루프를 돌아 yield 2를 만나 상태를 보존한 채 2를 내보냅니다.
- 이 방식은 무한한 데이터를 다루더라도 메모리에는 항상 단 하나의 정수 객체만 로드되므로 안전한 실행 흐름을 보장합니다.

##### 제너레이터의 확장: 코루틴(Coroutine)
---
- 제너레이터가 yield를 사용하여 함수 실행을 일시 정지하고 원하는 시점에 다시 깨우는 이 독특한 동작 방식은 파이썬의 동시성 프로그래밍인 '코루틴'의 기반이 됩니다. 일반 함수가 호출되면 멈춤 없이 한 방향으로만 실행되고 종료되는 반면, 코루틴은 실행 도중 제어권을 외부로 넘겼다 복귀받으며 서로 협력하듯 주도적으로 실행 흐름을 주고받는 고도의 함수 동작 방식입니다.

### 3. 제너레이터 표현식과 리스트 컴프리헨션 비교
--- 
- 파이썬에서는 대량의 데이터를 가공할 때 리스트 컴프리헨션([])을 자주 사용하지만, 이는 결과물을 모두 메모리에 일괄 적재하는 단점이 있습니다. 
소괄호(())를 사용하는 '제너레이터 표현식'은 컴프리헨션의 편리한 문법 규칙을 그대로 유지하면서도, 지연 평가 구조를 적용하여 시스템의 메모리를 획기적으로 아끼는 도구입니다. 대형 데이터 구조에서의 메모리를 효율적으로 절약할 수 있습니다.

#### 3.1 제너레이터 표현식
- 문법적 규칙
    - 리스트 컴프리헨션의 대괄호[]를 소괄호()로 바꾸기만 하면 제어 엔진이 제너레이터 표현식으로 자동 식별합니다.
- 메모리 절약 효과
    - 대형 리스트 컴프리헨션은 데이터 개수가 늘어날수록 메모리 용량이 개수에 비례하여 선형적으로 증가하지만, 제너레이터 표현식은 데이터가 10만 개든 1억 개든 상관없이 항상 고정된 수백 바이트 수준의 작은 메모리 크기만을 유지하는 유연함을 보여줍니다.

In [ ]:
import sys

# 리스트 컴프리헨션 방식 (모든 데이터를 메모리에 곧바로 적재)
list_comprehension = [num ** 2 for num in range(100000)]

# 소괄호를 활용한 제너레이터 표현식 방식 (필요할 때만 생성하는 구조)
generator_expression = (num ** 2 for num in range(100000))

# 두 방식의 메모리 용량 비교 및 절약 효과 측정하기
list_size = sys.getsizeof(list_comprehension)
gen_size = sys.getsizeof(generator_expression)

print(f"리스트 컴프리헨션 메모리 용량: {list_size} 바이트")
print(f"제너레이터 표현식 메모리 용량: {gen_size} 바이트")

- **list_comprehension**은 대괄호 구조를 인지하는 순간 0부터 99999까지의 모든 제곱수 결과를 미리 연산하여 100,000개의 방을 가진 거대한 리스트를 메모리에 직접 생성하므로 대략 80만 바이트 이상의 메모리를 소모합니다.
- **generator_expression**은 소괄호 구조를 식별하는 순간 "요청이 오면 제곱 연산을 수행하겠다"라는 규칙 코드만 보관하므로, 단 100여 바이트 남짓한 고정 용량만 메모리에 적재되어 메모리를 절약합니다.

## 개념 요약 키워드
---
- **이터러블과 이터레이터:** 반복 가능한 전체 데이터 집합(이터러블)과 내부 포인터를 순차적으로 이동시켜 실제 요소를 하나씩 꺼내오는 도구(이터레이터)의 관계입니다.
- **수동 순회 제어:** iter()와 next() 내장 함수를 연동하여 반복문의 도움 없이 데이터를 한 줄씩 주도적으로 수집하는 구조입니다.
- **yield 키워드:** 함수의 실행을 원하는 위치에서 잠시 멈추고 결과값을 내보낸 후, 내부 상태를 메모리에 온전히 얼려두어 보존하는 제어 명령어입니다.
- **지연 평가:** 데이터를 미리 다 계산해서 메모리에 올려두지 않고, 호출되어 필요한 시점이 되었을 때 비로소 연산을 시작하여 결과값을 생성하는 리소스 최적화 기법입니다.
- **제너레이터 표현식:** 소괄호() 문법 구조를 적용하여 리스트 컴프리헨션의 편리한 표현 방식을 계승하되, 고정된 최소 용량의 메모리만 점유하여 뛰어난 메모리 절약 효과를 제공하는 데이터 전달 방식입니다.

## 용어 사전 
---
| 용어 | 영어 표기 | 설명 |
|---|---|---|
| 이터러블 | Iterable | 리스트, 튜플, 딕셔너리, 문자열처럼 내부 알맹이들을 하나씩 차례대로 꺼내어 반복(순회)할 수 있는 모든 객체를 뜻합니다. |
| 이터레이터 | Iterator | 이터러블 객체로부터 생성되며, 실제 데이터를 가리키는 포인터를 한 칸씩 이동시키면서 다음 원소를 순차적으로 꺼내오는 제어 객체입니다. |
| `__iter__()` | `__iter__()` Magic Method | 이터러블 객체가 "나를 순회해 줄 전용 이터레이터를 만들어 달라"고 요청할 때 가동되는 파이썬의 특수 매직 메서드입니다. |
| `__next__()` | `__next__()` Magic Method | 이터레이터 객체가 다음 데이터가 있는 칸으로 위치를 한 단계 이동하고, 그 자리에 있는 실제 데이터 값을 추출하여 밖으로 반환하는 특수 매직 메서드입니다. |
| 제너레이터 | Generator | 대량의 데이터를 메모리에 한 번에 무겁게 올리지 않고, 필요할 때마다 값을 실시간으로 하나씩 생성해서 던져주는 특수한 형태의 루프 함수입니다. |
| `yield` 키워드 | `yield` Keyword | 제너레이터 함수 내부에서 값을 외부로 던질 때 쓰는 명령어입니다. 일반 함수의 `return`과 달리 값을 반환한 후 함수의 실행 상태와 메모리를 파괴하지 않고 그 자리에 그대로 일시 정지시켜 둡니다. |
| 지연 평가 | Lazy Evaluation | 대량의 데이터를 미리 전부 계산해서 메모리에 채워두는 것이 아니라, 데이터가 정말로 필요한 순간(호출되는 시점)이 되어서야 비로소 하나씩 연산하여 값을 만들어내는 메모리 최적화 기술입니다. |
| `StopIteration` 예외 | `StopIteration` Exception | 이터레이터나 제너레이터에서 더 이상 꺼낼 데이터가 존재하지 않을 때, 반복 순회가 안전하게 끝났음을 파이썬 시스템에 알리기 위해 자동으로 발생하는 정상적인 예외 알림 현상입니다. |

#
---

# 수업 내용
----

1. 이터러블(Iterable)
- 반복 가능한 객체(Object), 반복할 수 있는 대상
- \_\_iter\_\_(..) 정의되어 있다면 반복 가능!
- \_\_next\_\_(..) 인덱스를 한칸씩 이동하면서 요소를 꺼내오는 기능을 정의

In [2]:
# 이터러블 예시
nums = [10,20,30] # 리스트

nums.__iter__

<method-wrapper '__iter__' of list object at 0x000001E5C57B5780>

In [ ]:
for num in nums : 
    print(num)

10
20
30


2. 이터레이터(Iterator)
- 하나씩 꺼내는 도구, 반복자
- 실제로 값을 하나씩 꺼내며 현재 위치를 기억하는 객체
- 반복을 하려면 -> 이터레이터 객체 생성 (\_\_iter\_\_ : 호출 iter(...))
- 순서를 한칸씩 이동하면서 요소 조회 (\_\_next\_\_ : 호출 next(...))

In [ ]:
# 예시
fruits = ['apple','orange','melon','banana']   # fruits : 이터러블

it = iter(fruits)  # __iter__(..)호출   # iter : 함수 / iter() :함수 호출 
it                 # iter(fruits)가 만들어서 돌려준 이터레이터

In [ ]:
next(it)  # next : 함수 / next() : 이터레이터에서 다음 값을 하나 꺼내달라는 함수 호출

'apple'

In [6]:
next(it)

'orange'

In [7]:
next(it)

'melon'

In [8]:
next(it)

'banana'

In [ ]:
next(it)   # stopIteration # 예외 발생!!! 반복하는데, 다 꺼내왔으므로 "반복 다끝났어!" 라는 알림 신호!

StopIteration: 

3. 제너레이터(Generator)
- 값을 하나씩 만들어내는 특별한 이터레이터
- 이터레이터의 일종
- 함수의 실행을 정지하고, 원할 때 다시 재개하는 형태

In [10]:
# 예시
def generate_nums():

    yield 10
    yield 20
    yield 30

In [12]:
gen = generate_nums()
gen

<generator object generate_nums at 0x000001E5C5839F30>

In [13]:
next(gen)

10

In [14]:
next(gen)

20

In [15]:
next(gen)

30

In [16]:
next(gen)

StopIteration: 

In [ ]:
# 예시 2
def generate_nums():

    print("1번째")
    yield 10            # 함수의 실행이 중지됨

    print("2번쨰")
    yield 20

    print("3번째")
    yield 30

In [18]:
gen = generate_nums()
gen

<generator object generate_nums at 0x000001E5C55EE2C0>

In [ ]:
next(gen)    # next() : next함수를 호출하면, 결과값이 마지막 10에서 멈추고 다음 "2번째"를 호출하지 않아요~이게 제너레이터 작동이 멈춘 증거!

1번째


10

In [20]:
next(gen)

2번쨰


20

In [21]:
next(gen)

3번째


30

In [22]:
next(gen)

StopIteration: 

4. 제너레이터 표현식
- 값을 한꺼번에 만들지 않고, 필요할 때 하나씩 만들어내는 짧은 문법
- 참고 : 리스트 컴프리헨션 -> 생성 결과가 처음부터 메모리에 적재
- 리스트 대신, 튜플로 리스트 컴프리헨션과 동일하게 사용하면 된다. -> 메모리 소비가 적다
- 대용량 데이터 처리 시, 사용합니다.

    - 리스트 컴프리헨션 → 결과를 한꺼번에 만들어 저장
    - 제너레이터 표현식 → 결과를 미리 다 만들지 않음, 필요할 때 하나씩 생성


In [26]:
# 예시
gen = (i for i in range(1,101) if i % 2 == 0) # 1 ~ 101 중 짝수만 가져오는 제너레이터가 만들어졌음!!
gen

<generator object <genexpr> at 0x000001E5C57A9E50>

In [ ]:
next(gen)  # range로 정의한 숫자들 중, 하나의 숫자를 꺼내서 출력하고 멈춤!!!

2

In [ ]:
for num in gen :  # 위 next(gen)이 2 값을 호출하고 멈췄기 때문에, 다음 for문을 실행하면 4 값부터 호출하는거!
    print(num)

4
6
8
10
12
14
16
18
20
22
24
26
28
30
32
34
36
38
40
42
44
46
48
50
52
54
56
58
60
62
64
66
68
70
72
74
76
78
80
82
84
86
88
90
92
94
96
98
100


## [별첨.] 동기 vs 비동기
- 동기
    - 순서대로 실행하고 결과도 순서대로 도출
    - (eg.) A 작업 -> B 작업 -> C 작업 
            각 작업의 순차대로 시작되고 종료되어서, 앞선 작업이 종료되지 않으면 다음 작업이 진행되지 않습니다.
- 비동기
    - 순서대로 작업을 시작하지만, 결과는 먼저 처리된 순서대로 나오는 방식

# [이해하기!]
---
1. 객체란?
- Python에서는 우리가 다루는 거의 모든 값을 객체(object)
- 데이터와 그 데이터가 할 수 있는 기능을 하나로 묶어둔 것


맥락 이해하기
- " 이터레이터는 반복을 진행하는 구조, 제너레이터는 값을 필요할 때 하나씩 만들어내는 특별한 이터레이터"
- 개념
    - 이터레이터 → 반복을 실제로 진행하는 방식
    - 제너레이터 → 이터레이터를 쉽게 만들고, 값을 필요할 때 하나씩 생성하는 방식 # 특별한 이터레이터!!!!
- 왜 이터레이터를 쓰는가?
    - 객체의 정보를 기억하면서 하나씩 꺼내주는 역할
    - 이터레이터는 반복을 자동으로 진행하기 위해 필요
- 왜 제너레이터를 쓰는가? # 가장 중요한 이유와 목적은 "메모리 절약!!!!"
    - 이터레이터의 일종인데, 메모리를 아끼면서 "필요할 때 작업을 하나씩" 하게 하는 역할
    - 제너레이터는 값을 미리 전부 만들지 않고, 필요할 때 하나씩 만들기 위해 사용
- 이터레이터와 제너레이터의 목적!
    - 이터레이터 : 반복 위치를 기억하고 다음 값을 준다!
    - 제너레이터 : 값이 필요할 때 생성하면서 다음 값을 준다!

---
2. 동기 VS 비동기

맥락 이해하기
- " 동기와 비동기는 일을 처리할 때 다음 작업이 이전 작업을 기다리느냐의 차이 "
- 동기 : 앞 일이 끝나야 다음 일을 한다.
→ 앞의 일이 끝날 때까지 기다린다
→ 끝나면 다음 일을 한다

- 비동기 : 기다리는 동안 다른 일을 할 수 있다. / 비동기가 무조건 더 빠른 건 아닙니다. / 동시에 여러 일을 합니다./ 비동기는 특히 “기다림이 많은 작업”에서 효과가 큽니다.
→ 준비되면 다시 돌아와 처리한다

